In [1]:
import numpy as np
import pandas as pd

from PyFairnessAI.model_selection import cross_val_score_fairness
from PyFairnessAI.data import privileged_groups_sens, unprivileged_groups_sens

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression

from aif360.datasets import GermanDataset

pip install 'aif360[inFairness]'
pip install 'aif360[OptimalTransport]'
pip install 'aif360[FairAdapt]'
pip install 'aif360[LFR]'


In [2]:
german_data_aif = GermanDataset(
        protected_attribute_names=['age'],            
        privileged_classes=[lambda x: x >= 25],      
        features_to_drop=['personal_status', 'sex'] 
    )

# Cambiar los labels de la respuesta de 2 a 0 (label 2 es desfavorable y 1 es favorable) 
german_data_aif.labels[german_data_aif.labels.ravel() == 2] =  german_data_aif.labels[german_data_aif.labels.ravel() == 2] - 2
german_data_aif.unfavorable_label = german_data_aif.unfavorable_label - 2

response_name = german_data_aif.label_names[0]
response_favorable_label = german_data_aif.favorable_label 
response_unfavorable_label = german_data_aif.unfavorable_label 
sens_variable_name = german_data_aif.protected_attribute_names[0]
sens_privileged_groups = [privileged_groups_sens(german_data_aif)]
sens_unprivileged_groups = [unprivileged_groups_sens(german_data_aif)]

print('response_name:', response_name)
print('response_favorable_label:', response_favorable_label)
print('response_unfavorable_label:', response_unfavorable_label)
print('sens_variable_name:', sens_variable_name)
print('sens_privileged_groups:', sens_privileged_groups)
print('sens_unprivileged_groups:', sens_unprivileged_groups)

response_name: credit
response_favorable_label: 1.0
response_unfavorable_label: 0.0
sens_variable_name: age
sens_privileged_groups: [{'age': 1}]
sens_unprivileged_groups: [{'age': 0}]


In [3]:
data_arr = np.concatenate([german_data_aif.labels, german_data_aif.features], axis=1)
data_col_names = [response_name] + german_data_aif.feature_names

german_data_pd = pd.DataFrame(data=data_arr, columns=data_col_names)
# The sensitive variable/s must index the df in order to avoid problems with some aif360 processors (PostProcessingMetaEstimator)
german_data_pd.index = german_data_pd[sens_variable_name] 
german_data_pd.head(6)

,credit,month,credit_amount,investment_as_income_percentage,residence_since,age,number_of_credits,people_liable_for,status=A11,status=A12,...,housing=A152,housing=A153,skill_level=A171,skill_level=A172,skill_level=A173,skill_level=A174,telephone=A191,telephone=A192,foreign_worker=A201,foreign_worker=A202
age,,,,,,,,,,,,,,,,,,,,,
1.0,1.0,6.0,1169.0,4.0,4.0,1.0,2.0,1.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
0.0,0.0,48.0,5951.0,2.0,2.0,0.0,1.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
1.0,1.0,12.0,2096.0,2.0,3.0,1.0,1.0,2.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1.0,1.0,42.0,7882.0,2.0,4.0,1.0,1.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
1.0,0.0,24.0,4870.0,3.0,4.0,1.0,2.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
1.0,1.0,36.0,9055.0,2.0,4.0,1.0,1.0,2.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0


In [4]:
Y = german_data_pd[response_name]
X = german_data_pd[german_data_aif.feature_names] # X = german_data_pd[quant_predictors + cat_predictors]
A = german_data_pd[sens_variable_name] # sensitive variable / protected attribute

In [5]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.85, random_state=123, stratify=Y)
inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=111)
log_reg_estimator = LogisticRegression(solver='liblinear', random_state=123)

In [6]:
fairness_metrics = ['statistical_parity_difference', 'abs_statistical_parity_difference',
                    'disparate_impact_ratio', 'abs_equal_opportunity_difference',
                    'average_odds_error', 'false_positive_rate_difference', 
                    'false_negative_rate_difference', 'true_positive_rate_difference',
                    'true_negative_rate_difference', 'false_positive_rate_ratio',
                    'false_negative_rate_ratio', 'true_positive_rate_ratio', 
                    'true_negative_rate_ratio', 'positive_predicted_value_difference',
                    'positive_predicted_value_ratio', 'positive_predicted_value_abs_difference']

In [7]:
final_metric, metric_iters = {}, {}

for metric in fairness_metrics:

    final_metric[metric] , metric_iters[metric] = cross_val_score_fairness(estimator=log_reg_estimator, X=X_train, y=Y_train, 
                                                     sens_variable_name=sens_variable_name, 
                                                     priv_group=sens_privileged_groups[0][sens_variable_name],
                                                     pos_label=response_favorable_label, 
                                                     scoring=metric, 
                                                     cv=inner)

-0.2552447552447552
-0.10380881254667662
-0.1019955654101995
0.22834645669291342
-0.2750977835723599


In [8]:
final_metric

{'statistical_parity_difference': -0.2473340425963781,
 'abs_statistical_parity_difference': 0.2473340425963781,
 'disparate_impact_ratio': 0.699935978204915,
 'abs_equal_opportunity_difference': 0.23193257477427717,
 'average_odds_error': 0.25085774835063585,
 'false_positive_rate_difference': -0.16024718409857377,
 'false_negative_rate_difference': 0.23193257477427714,
 'true_positive_rate_difference': -0.23193257477427714,
 'true_negative_rate_difference': 0.16024718409857377,
 'false_positive_rate_ratio': 0.7700054112554112,
 'false_negative_rate_ratio': 7.135734265734266,
 'true_positive_rate_ratio': 0.7481137107111308,
 'true_negative_rate_ratio': 1.483767633369137,
 'positive_predicted_value_difference': -0.10156009201621556,
 'positive_predicted_value_ratio': 0.8805471600979716,
 'positive_predicted_value_abs_difference': 0.19289867469338093}

In [9]:
metric_iters

{'statistical_parity_difference': [-0.03852401949408213,
  -0.13546288080449576,
  -0.4527324527324527,
  -0.4035626535626536,
  -0.20638820638820632],
 'abs_statistical_parity_difference': [0.03852401949408213,
  0.13546288080449576,
  0.4527324527324527,
  0.4035626535626536,
  0.20638820638820632],
 'disparate_impact_ratio': [0.9485111662531017,
  0.8066694807935837,
  0.4736525143029208,
  0.5297065139584825,
  0.741140215716487],
 'abs_equal_opportunity_difference': [0.0980392156862746,
  0.08127721335268512,
  0.4474358974358974,
  0.21904761904761905,
  0.3138629283489096],
 'average_odds_error': [0.1542319630554925,
  0.09673616765195231,
  0.3679487179487179,
  0.4467331118493909,
  0.18863878124762554],
 'false_positive_rate_difference': [0.21042471042471045,
  -0.1121951219512195,
  -0.28846153846153844,
  -0.6744186046511628,
  0.06341463414634141],
 'false_negative_rate_difference': [0.0980392156862745,
  0.08127721335268506,
  0.44743589743589746,
  0.21904761904761905,
 